# sol07: Tokenizer Greedy Longest Match

Contains:
- the same scenario as `07_mock`
- one complete reference implementation
- grading tests


In [ ]:
from typing import Any

VOCAB = {
    "app": 1,
    "apple": 2,
    "pie": 3,
    "pi": 4,
    "UNK": -1,
}


In [ ]:
def tokenize_longest(text: str, vocab: dict[str, int], compress_unk: bool = False) -> list[int]:
    if "UNK" not in vocab:
        raise ValueError("vocab_missing_UNK")

    unk_id = vocab["UNK"]
    token_lengths = [len(token) for token in vocab if token != "UNK"]
    max_len = max(token_lengths) if token_lengths else 0

    tokens: list[int] = []
    i = 0
    while i < len(text):
        matched_id: int | None = None
        matched_len = 0

        upper = min(len(text), i + max_len)
        for j in range(upper, i, -1):
            candidate = text[i:j]
            if candidate in vocab and candidate != "UNK":
                matched_id = vocab[candidate]
                matched_len = j - i
                break

        if matched_id is None:
            matched_id = unk_id
            matched_len = 1

        if compress_unk and matched_id == unk_id and tokens and tokens[-1] == unk_id:
            i += matched_len
            continue

        tokens.append(matched_id)
        i += matched_len

    return tokens


def tokenize_batch(texts: list[str], vocab: dict[str, int], compress_unk: bool = False) -> list[list[int]]:
    return [tokenize_longest(text, vocab, compress_unk=compress_unk) for text in texts]


In [ ]:
def run_exam07_tests() -> None:
    assert tokenize_longest("apple", VOCAB) == [2]
    assert tokenize_longest("apppie", VOCAB) == [1, 3]
    assert tokenize_longest("bbb", VOCAB) == [-1, -1, -1]
    assert tokenize_longest("bbb", VOCAB, compress_unk=True) == [-1]
    assert tokenize_longest("appbbbapp", VOCAB, compress_unk=True) == [1, -1, 1]

    custom_vocab = {"a": 7, "ab": 8, "abc": 9, "UNK": -1}
    assert tokenize_longest("abcabx", custom_vocab) == [9, 8, -1]

    batch = tokenize_batch(["apple", "bbb", "apppie"], VOCAB, compress_unk=True)
    assert batch == [[2], [-1], [1, 3]]

    try:
        tokenize_longest("abc", {"a": 1})
        raise AssertionError("Expected vocab_missing_UNK")
    except ValueError as exc:
        assert "vocab_missing_UNK" in str(exc)

    print("07_mock tests passed")


run_exam07_tests()


## Walkthrough: Exactly How to Solve `07_mock`

### 0) First 2 minutes
- Write the greedy rule in a comment:
  - longest match at index
  - fallback to `UNK` and advance one char
- Confirm `UNK` is mandatory.

### 1) Should I read tests now?
Yes:
- `apple -> [2]` proves longest token priority.
- `bbb` with compression gives one `-1`.
- custom vocab test validates greedy at each step.
- missing `UNK` must raise `vocab_missing_UNK`.

### 2) Coding order
1. Validate `UNK` exists.
2. Precompute max token length (excluding `UNK`).
3. Implement greedy scan in `tokenize_longest`.
4. Add compression logic.
5. Implement `tokenize_batch` as list comprehension.

### 3) One concrete example to narrate aloud
`apppie` with vocab `app=1, pie=3`:
- index 0 longest match is `app` -> `1`
- index 3 longest match is `pie` -> `3`
- output `[1,3]`

### 4) What to say while coding
- "I’m implementing greedy longest-match with bounded window length."
- "I scan from longest to shortest candidate at each index."
- "Compression is optional post-processing and does not change base matching semantics."

### 5) Self-check before final run
- Do I ever skip characters incorrectly?
- Does compression only collapse consecutive UNKs?
- Is batch wrapper behavior identical to single-string behavior?
